In [ ]:
pip install lightning timm torchmetrics

: 

In [ ]:
pip install grad-cam

In [ ]:
import sys
import os
import importlib.util

# 1. Define the absolute path
PROJECT_PATH = '/content/drive/MyDrive/AetherVision'
os.chdir(PROJECT_PATH)

# 2. Define the exact file paths
data_loader_path = os.path.join(PROJECT_PATH, 'src', 'data_loader.py')
model_path = os.path.join(PROJECT_PATH, 'src', 'model.py')

# 3. Helper to load a module from a specific file path
def load_module_from_path(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

# 4. Load the modules manually
data_loader = load_module_from_path('src.data_loader', data_loader_path)
model_module = load_module_from_path('src.model', model_path)

# 5. Extract classes
WeatherDataset = data_loader.WeatherDataset
AetherModel = model_module.AetherModel

print("Imports forced successfully!")

In [ ]:
import sys
import os

# Set the path
PROJECT_PATH = '/content/drive/MyDrive/AetherVision'
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# Standard import
from src.data_loader import WeatherDataset
from src.model import AetherModel

print("Imports successful!")

In [ ]:
# --- 1. Imports and Setup ---
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import transforms

# Internal project imports
from src.data_loader import WeatherDataset
from src.model import AetherModel

In [ ]:
# --- 2. Data Exploration & Sanity Check ---
# 1. Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 2. Initialize
dataset = WeatherDataset(root_dir="./data/raw", split='train', transform=transform)

# 3. Create a reverse mapping for display
idx_to_weather = {v: k for k, v in dataset.weather_to_idx.items()}

# 4. Improved display function
def show_samples(ds, n=5):
    fig, axes = plt.subplots(1, n, figsize=(15, 5))
    for i in range(n):
        img, label_idx = ds[i]
        
        # Move to CPU to ensure plotting works safely
        # .detach() ensures we don't track gradients for plotting
        img_plot = img.detach().cpu().permute(1, 2, 0)
        
        axes[i].imshow(img_plot)
        axes[i].set_title(f"Label: {idx_to_weather[label_idx]}")
        axes[i].axis('off')
    plt.show()

show_samples(dataset)

In [ ]:
# Check unique labels found
unique_labels = set(dataset.labels.values())
print(f"Unique labels found in dataset: {unique_labels}")

# Check count of each
from collections import Counter
print(Counter(dataset.labels.values()))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Use the pre-computed label list instead of iterating over the dataset
label_counts = {k: 0 for k in dataset.weather_to_idx.keys()}
for val in dataset.labels.values():
    weather_name = [k for k, v in dataset.weather_to_idx.items() if v == val][0]
    label_counts[weather_name] += 1

# Plot
plt.figure(figsize=(8, 5))
sns.barplot(x=list(label_counts.keys()), y=list(label_counts.values()), palette='viridis')
plt.title("Weather Class Distribution (Training Set)")
plt.ylabel("Number of Images")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# 1. Efficiently get counts without loading images
# This takes milliseconds and uses 0 GPU
label_counts = Counter(dataset.labels.values())

# 2. Map indices to names based on what is actually present
idx_to_name = {v: k for k, v in dataset.weather_to_idx.items()}
names = [idx_to_name[idx] for idx in label_counts.keys()]
counts = list(label_counts.values())

# 3. Plot
plt.figure(figsize=(8, 5))
sns.barplot(x=names, y=counts, palette='viridis')
plt.title("Weather Class Distribution (Training Set)")
plt.ylabel("Number of Images")
plt.xlabel("Weather Category")
plt.show()

# 4. Final verification
print(f"Dataset length: {len(dataset)}")
print(f"Weather mapping: {dataset.weather_to_idx}")

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os

class InferenceDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.images = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.images[idx] # Returns image and filename instead of label

# Usage:
inf_dataset = InferenceDataset(image_dir="./data/raw/test_dataset/test_images", transform=transform)
inf_loader = torch.utils.data.DataLoader(inf_dataset, batch_size=32)

In [ ]:
import timm
import torch

# Load a ViT model pre-trained on ImageNet and adapt it for 3 classes
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=3)
model.to('cuda')
model.eval()

In [ ]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"CUDA Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np
import torch

# 1. Define the reshape function for ViT
def vit_reshape_transform(tensor, height=14, width=14):
    # Skip the CLS token
    result = tensor[:, 1:, :]
    result = result.reshape(result.size(0), height, width, result.size(2))
    # Bring the channel dimension to the first dimension
    result = result.transpose(2, 3).transpose(1, 2)
    return result

# 2. Setup Grad-CAM for your ViT model
# Target the last normalization layer of the last transformer block
target_layers = [model.blocks[-1].norm1]

cam = GradCAM(
    model=model,
    target_layers=target_layers,
    reshape_transform=vit_reshape_transform
)

# 3. Generate visualization for an image
# Assuming 'input_tensor' is your preprocessed image (batch_size, 3, 224, 224)
targets = [ClassifierOutputTarget(0)] # Target class (e.g., 0=Cloudy)
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)

# 4. Overlay and show
rgb_img = input_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy()
# Normalize for display
rgb_img = (rgb_img - rgb_img.min()) / (rgb_img.max() - rgb_img.min())

visualization = show_cam_on_image(rgb_img, grayscale_cam[0, :], use_rgb=True)

import matplotlib.pyplot as plt
plt.imshow(visualization)
plt.axis('off')
plt.show()